# Qwen3 1.7B Dev Baseline

Closed-book baseline evaluation for `Qwen/Qwen3-1.7B` on the dev preview benchmark.

## Setup

Run this in Colab. For this public model you should not need a Hugging Face API key. If Hugging Face rate limits you or you switch to a gated model later, then log in with `huggingface_hub.login()` or set `HF_TOKEN` in Colab secrets.

In [ ]:
!pip install -q -U transformers accelerate pandas tqdm

## Imports

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

## Locate Repo

If you cloned the repo somewhere else in Colab, change `PROJECT_ROOT` below.

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "benchmark").exists():
    candidate = Path("/content/SC_fine-tune")
    if candidate.exists():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / "benchmark").exists(), "Set PROJECT_ROOT to the cloned repo directory."
PROJECT_ROOT

## Load Dev Records

In [ ]:
dev_dir = PROJECT_ROOT / "benchmark" / "data" / "dev"
paths = sorted(dev_dir.glob("*_preview.jsonl"))

records = []
for path in paths:
    with path.open() as f:
        for line in f:
            records.append(json.loads(line))

len(records), [p.name for p in paths]

## Prompt Format

This is closed-book: the model gets the problem only, no theorem sheet and no examples.

In [ ]:
def make_prompt(problem):
    return (
        "Solve the following stochastic-process problem. "
        "Return only the requested JSON inside the answer tags.\n\n"
        f"{problem}\n\n"
        "Final answer:\n<answer>"
    )

## Load Model

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()

device = next(model.parameters()).device
device

## Generation

In [ ]:
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0

def generate_answer(problem):
    messages = [{"role": "user", "content": make_prompt(problem)}]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Answer Extraction and Scoring

In [ ]:
def extract_json_object(text):
    tag_match = re.search(r"<answer>\s*(.*?)\s*</answer>", text, flags=re.S)
    if tag_match:
        text = tag_match.group(1)

    json_match = re.search(r"\{.*?\}", text, flags=re.S)
    if not json_match:
        return None

    try:
        return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        return None

def normalize_answer(answer):
    if answer is None:
        return None
    return {str(k): str(v).lower() if isinstance(v, bool) else str(v) for k, v in answer.items()}

def is_correct(predicted, canonical):
    return normalize_answer(predicted) == normalize_answer(canonical)

## Run Eval

This is only 60 records, so a simple loop is fine.

In [ ]:
rows = []

for record in tqdm(records):
    raw_output = generate_answer(record["problem"])
    predicted = extract_json_object(raw_output)
    correct = is_correct(predicted, record["canonical_answer"])
    metadata = record.get("metadata", {})

    rows.append({
        "id": record["id"],
        "family": record["family"],
        "problem_type": record["problem_type"],
        "difficulty": record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": record["problem"],
        "canonical_answer": record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "correct": correct,
    })

df = pd.DataFrame(rows)
df.head()

## Metrics

In [ ]:
overall = df["correct"].mean()
print(f"Overall accuracy: {overall:.3f} ({df['correct'].sum()}/{len(df)})")

display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())

## Inspect Mistakes

In [ ]:
mistakes = df[~df["correct"]].copy()
mistakes[["id", "family", "problem_type", "difficulty", "canonical_answer", "predicted_answer", "raw_output"]].head(20)

## Save Results

In [ ]:
result_dir = PROJECT_ROOT / "results" / "baselines" / "qwen3_1_7b_dev_closed_book"
result_dir.mkdir(parents=True, exist_ok=True)

outputs_path = result_dir / "outputs.jsonl"
metrics_path = result_dir / "metrics.json"
csv_path = result_dir / "outputs.csv"

with outputs_path.open("w") as f:
    for row in rows:
        f.write(json.dumps(row, sort_keys=True) + "\n")

df.to_csv(csv_path, index=False)

metrics = {
    "model_name": MODEL_NAME,
    "prompt_mode": "closed_book",
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "num_records": int(len(df)),
    "num_correct": int(df["correct"].sum()),
    "accuracy": float(df["correct"].mean()),
}

with metrics_path.open("w") as f:
    json.dump(metrics, f, indent=2, sort_keys=True)

outputs_path, metrics_path, csv_path